In [3]:
import sys
sys.path.insert(0, '..')  # repo root, so downstream_processing can be imported
import downstream_processing as sv
from cyvcf2 import VCF
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
sv_dict = sv.base.load_dict(folder='processed/nanoseq', prefix='')

FileNotFoundError: [Errno 2] No such file or directory: 'processed/nanoseq/0dQ6/tmpNanoSeq/post/results.muts.tsv'

In [56]:
?sv.base.load_dict

Signature:
sv.base.load_dict(
    folder='./test_conservative_noTGCA',
    prefix='test_out',
    simple=False,
)
Docstring:
Loads mutation data from multiple samples within a given folder.

Parameters:
----------
folder : str, optional
    Path to the folder containing mutation files (default: "./test_conservative_noTGCA").
prefix : str, optional
    Prefix for identifying relevant sample files (default: "test_out").
simple : bool, optional
    If True, loads only pyrkeys from each sample file; otherwise, loads full mutation data (default: True).

Returns:
-------
dict
    Dictionary mapping sample file names to their corresponding mutation data (either a DataFrame or a list of IDs).
File:      ~/Apis/somatic/docker/somvar/base.py
Type:      function

In [9]:
for key,item in sv_dict.items():
    j = item
    j['sample']=key
fulldf = pd.concat([item for _,item in sv_dict.items()])

In [11]:
fulldf.to_csv('../data/somdata_all.csv')

In [13]:
doubleID, doubleINFO = sv.double.get_double_sites(sv_dict)

In [16]:
doubleID

,sample,CHROM,POS,UMI,ID
0,test_out_0dD3,NC_037638.1,14527404,GAT|GTA,NC_037638.1:14527404
1,test_out_0dD3,NC_037638.1,14527418,GAT|GTA,NC_037638.1:14527418
4,test_out_0dD3,NC_037638.1,9795648,AGC|TAG,NC_037638.1:9795648
5,test_out_0dD3,NC_037638.1,9795649,AGC|TAG,NC_037638.1:9795649
8,test_out_0dD3,NC_037639.1,13180892,TAT|ACC,NC_037639.1:13180892
...,...,...,...,...,...
3271,test_out_nQ1,NW_020555859.1,28660,CCT|AAC,NW_020555859.1:28660
3272,test_out_nQ1,NW_020555859.1,28600,CAG|GGA,NW_020555859.1:28600
3273,test_out_nQ1,NW_020555859.1,28644,CAG|GGA,NW_020555859.1:28644
3284,test_out_nQ1,NW_020555870.1,32491,CAT|CGT,NW_020555870.1:32491


In [26]:
fulldf['double'] = [1 if i in set(list(doubleID.ID)) else 0 for i in list(fulldf.ID)]

In [30]:
vcf_file = "/home/taliador/1_mellifera/raw_data/aligned/sorted/RG/merge/haplotype/BetterB_Batch1_full_geno.vcf.gz"


IDs,filtered_germ_sites,somvar_filtgerm =  sv.germ.filter_germ(somvar_dict=sv_dict, vcf_file=vcf_file, threads=30)

In [33]:
fulldf['germ'] = [1 if i in IDs else 0 for i in list(fulldf.ID)]

In [36]:
fulldf.germ.sum()

204

In [37]:
fulldf

,chrom,chromStart,context,commonSNP,shearwater,bulkASXS,bulkNM,bulkForwardA,bulkForwardC,bulkForwardG,...,pyrsub,stdsub,ismasked,dplxBarcode,pyrkey,ID,POS,sample,double,germ
0,NC_037638.1,14371590,AGT,0,0,110,1,0,0,621,...,ACT>T,AGT>A,0,ACT|TGT,"NC_037638.1,14371590,ACT>T",NC_037638.1:14371590,14371591,test_out_0dD3,0,0
1,NC_037638.1,14527404,TCG,0,0,122,0,0,458,0,...,TCG>T,TCG>T,0,GAT|GTA,"NC_037638.1,14527404,TCG>T",NC_037638.1:14527404,14527405,test_out_0dD3,1,0
2,NC_037638.1,14527418,CGT,0,0,122,0,0,0,467,...,ACG>T,CGT>A,0,GAT|GTA,"NC_037638.1,14527418,ACG>T",NC_037638.1:14527418,14527419,test_out_0dD3,1,0
3,NC_037638.1,27253962,AGA,0,0,143,0,0,0,1564,...,TCT>T,AGA>A,0,AAT|AAT,"NC_037638.1,27253962,TCT>T",NC_037638.1:27253962,27253963,test_out_0dD3,0,0
4,NC_037638.1,9795648,CAA,0,0,111,1,760,0,1,...,TTG>C,CAA>G,0,AGC|TAG,"NC_037638.1,9795648,TTG>C",NC_037638.1:9795648,9795649,test_out_0dD3,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329,NW_020555870.1,39083,ACG,0,0,78,2,0,14,0,...,ACG>T,CGT>A,0,TCT|TTT,"NW_020555870.1,39083,ACG>T",NW_020555870.1:39083,39084,test_out_nQ1,0,1
330,NW_020555870.1,39083,ACG,0,0,78,2,0,14,0,...,ACG>T,CGT>A,0,GAT|GAC,"NW_020555870.1,39083,ACG>T",NW_020555870.1:39083,39084,test_out_nQ1,0,1
331,NW_020555870.1,39083,ACG,0,0,78,2,0,14,0,...,ACG>T,CGT>A,0,TGT|CAG,"NW_020555870.1,39083,ACG>T",NW_020555870.1:39083,39084,test_out_nQ1,0,1
332,NW_020555870.1,39083,ACG,0,0,78,2,0,14,0,...,ACG>T,CGT>A,0,ACC|TTT,"NW_020555870.1,39083,ACG>T",NW_020555870.1:39083,39084,test_out_nQ1,0,1


In [39]:
fulldf.germ.sum()

204

In [41]:
fulldf["newID"] = [f'{k.chrom}:{k.chromStart+1}' for i,k in fulldf.iterrows()]

In [46]:
fulldf["germ"]  = [1 if i in IDs else 0 for i in list(fulldf.newID)]

In [45]:
np.sum(newgerm)

1918

In [47]:
fulldf.double.sum()

986

In [49]:
fulldf["germ"].sum()

1918

In [50]:
fulldf.shape[0]

2055

In [51]:
fulldf["germ"].sum()/fulldf.shape[0]

0.9333333333333333

In [52]:
fulldf.to_csv('../data/somatic_muts_with_germ_and_double.csv')

In [54]:
def load_discard_csv(sample_folder,
                     file_loc="tmpNanoSeq/post/discardedvariants.csv",
                     simple=True):
    """
    Loads a list of discarded somatic variants from a CSV file and optionally returns them in a simplified format.
    
    Parameters:
    sample_folder (str): Path to the folder containing the discard file.
    file_loc (str, optional): Relative path to the discard file within the sample folder. 
                              Defaults to "tmpNanoSeq/post/discardedvariants.csv".
    simple (bool, optional): If True, returns a list of variant site identifiers in the format "chromosome:position".
                             If False, returns the full DataFrame. Defaults to True.
    
    Returns:
    list or pandas.DataFrame:
        - If `simple=True`, returns a list of discarded variant site identifiers (e.g., ["chr1:12345", "chr2:67890"]).
        - If `simple=False`, returns a pandas DataFrame containing the full discarded variant information.
    """
    discard_file = sample_file = os.path.join(sample_folder, file_loc)
    discard = pd.read_csv(sample_file)
    discard = discard.drop_duplicates(subset=["chrom", "chromStart"])
    if simple==False:
        return discard
    else:
        return [str(k.chrom)+':'+str(k.chromStart) for i,k in discard.iterrows()]

In [57]:
folder="./test_conservative_noTGCA"
prefix="test_out"
sv_discard = []
for i in [j for j in  os.listdir(folder) if j.startswith(prefix)]:
    
    l = os.path.join(folder, i)
    print(f"processing {l}")
    sv_discard.append(load_discard_csv(sample_folder=l, simple=False))
    print(f" done! ")


processing ./test_conservative_noTGCA/test_out_0dD3
 done! 
processing ./test_conservative_noTGCA/test_out_0dQ4
 done! 
processing ./test_conservative_noTGCA/test_out_0dW5
 done! 
processing ./test_conservative_noTGCA/test_out_4wD1
 done! 
processing ./test_conservative_noTGCA/test_out_6wW1
 done! 
processing ./test_conservative_noTGCA/test_out_nQ1
 done! 
